In [1]:
%reset -f
%load_ext autoreload
%autoreload 2
import sys
import sysconfig



In [2]:
import os
import sys
import time
from pathlib import Path
import json
import subprocess
import threading
import queue
import gc
import concurrent.futures
import sysconfig
import psutil  # Required: pip install psutil

# ================= HPC Configuration =================

# 1. Configure the parallel worker count (including hyperthreads)
# If you have 32 physical cores, you often get 64 logical threads. Set this higher to saturate the machine.
KERN_NUM = 8

# 2. Memory safety threshold (%)
# If system memory usage exceeds 90%, pause launching new tasks to avoid exhausting RAM.
MEM_SAFE_THRESHOLD = 90.0 

# 3. Timeout setting (seconds)
TIMEOUT_S = 600

# 4. Seed range
START_INDEX = 1
END_INDEX = 100

# ===============================================

# Print Python environment information
print("Python version:", sys.version)
print("HPC Mode: Enabled")
print(f"Target Core/Worker Count: {KERN_NUM}")

# Directory configuration
LOG_DIR = Path("Log")
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Seed argument parsing (supports command-line overrides)
# Handle Env Vars / CLI Args
env_max = os.getenv('BATCH_SEED_MAX')
env_start = os.getenv('BATCH_SEED_START')

arg_max = None
arg_start = None

# Check whether we are running inside a Jupyter/IPython environment
# In Jupyter, sys.argv[1] is often '-f', so we must ignore it.
is_jupyter = 'ipykernel' in sys.modules

if not is_jupyter:
    # Only parse sys.argv when running in a plain terminal.
    if len(sys.argv) >= 2: 
        # Validate that the argument is numeric so we do not accidentally parse an unrelated flag.
        if sys.argv[1].isdigit():
            arg_max = sys.argv[1]
    if len(sys.argv) >= 3: 
        if sys.argv[2].isdigit():
            arg_start = sys.argv[2]
else:
    print("[Info] Jupyter environment detected. Ignoring CLI arguments (sys.argv).")

# The parsing logic is now safe.
max_seed = int(arg_max or env_max or END_INDEX)
start_seed = int(arg_start or env_start or START_INDEX)

if start_seed > max_seed:
    raise SystemExit(f'start_seed ({start_seed}) cannot exceed max_seed ({max_seed})')

# Locate the notebook
ROOT = Path('.').resolve()
# Look for the main notebook in the repository root.
NB_CANDIDATES = [ROOT / 'MP-OPT.ipynb']
NB_PATH = next((p for p in NB_CANDIDATES if p.exists()), None)

if NB_PATH is None:
    raise FileNotFoundError('Cannot locate MP-OPT.ipynb in the repository root.')

TEMP_SCRIPT_PATH = Path(f"_temp_worker_script_hpc.py")

# === Step 1: Extract code while keeping the original workflow ===

def create_worker_script(nb_path, script_path):
    print(f"Extracting code from {nb_path} to {script_path}...")
    try:
        nb_content = nb_path.read_text(encoding='utf-8')
        nb = json.loads(nb_content)
    except Exception as e:
        print(f"Error reading notebook: {e}")
        sys.exit(1)
    
    code_lines = []
    
    # Header: force unbuffered I/O
    code_lines.append("import os, sys, json, traceback")
    code_lines.append("if hasattr(sys.stdout, 'reconfigure'):")
    code_lines.append("    sys.stdout.reconfigure(line_buffering=True)")
    code_lines.append("    sys.stderr.reconfigure(line_buffering=True)")
    
    
    for cell in nb.get('cells', []):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', ''))
        # Filter out IPython magic commands
        clean_source = [line for line in source.split('\n') if not line.lstrip().startswith(('%', '!'))]
        source_text = '\n'.join(clean_source)
        source_text = source_text.replace('use_original_modplants = True', 'use_original_modplants = False')
        if source_text.strip():
            code_lines.append(source_text + '\n')

    # Footer: capture the worker result
    footer = """
try:
    _res_feasible = locals().get('bfs_feasible', None)
    _res_path = locals().get('corpus_path', None)
    if _res_path: _res_path = str(_res_path)
    print(f"\\n__WORKER_RESULT__:{json.dumps({'status': 'OK', 'feasible': _res_feasible, 'corpus_path': _res_path}, ensure_ascii=False)}", flush=True)
except Exception as e:
    print(f"\\n__WORKER_RESULT__:{json.dumps({'status': f'ERROR: {e}', 'feasible': None, 'corpus_path': None})}", flush=True)
"""
    code_lines.append(footer)
    
    with open(script_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(code_lines))

create_worker_script(NB_PATH, TEMP_SCRIPT_PATH)

# === Step 2: Define the HPC worker logic ===

def wait_for_memory():
    """Block while memory usage is above the configured high-water mark."""
    while True:
        mem = psutil.virtual_memory()
        if mem.percent < MEM_SAFE_THRESHOLD:
            return
        print(f"[System] RAM usage high ({mem.percent}%). Waiting for resources...", end='\r')
        time.sleep(2)

def run_single_seed(seed):
    # 1. Memory check
    wait_for_memory()

    start_time = time.perf_counter()
    
    # 2. Set key environment variables to avoid multithreading conflicts
    # When launching many processes on an HPC machine, disable NumPy/Pandas internal multithreading,
    # otherwise 64 processes * 64 threads can overwhelm the machine.
    env = os.environ.copy()
    env['MODPLANT_SEED'] = str(seed)
    env['PYTHONUNBUFFERED'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    
    log_txt_path = LOG_DIR / f"seed_{seed}.log"
    
    status = 'UNKNOWN'
    bfs_feasible = None
    corpus_path = None
    
    # print(f"[Seed {seed}] Queued.", end='')  # Keep console output concise and avoid flooding the terminal.

    def _reader_thread(pipe, q):
        try:
            for line in iter(pipe.readline, ''):
                q.put(line)
        finally:
            q.put(None)

    try:
        with open(log_txt_path, 'w', encoding='utf-8') as log_file:
            # Reuse the current Python interpreter path for the worker subprocess.
            process = subprocess.Popen(
                [sys.executable, '-u', str(TEMP_SCRIPT_PATH)],
                env=env,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8',
                bufsize=1
            )

            q = queue.Queue()
            reader = threading.Thread(target=_reader_thread, args=(process.stdout, q), daemon=True)
            reader.start()

            timed_out = False

            while True:
                # Timeout check
                if time.perf_counter() - start_time > TIMEOUT_S:
                    timed_out = True
                    status = f"TIMEOUT (> {TIMEOUT_S}s)"
                    try:
                        process.terminate()  # Try a graceful termination first
                        process.wait(timeout=2)
                    except:
                        try:
                            process.kill()  # Force-kill the process if needed
                        except:
                            pass
                    break

                try:
                    line = q.get(timeout=0.2)
                except queue.Empty:
                    line = None

                if line is None:
                    if process.poll() is not None:
                        break
                    continue

                # Write to the log file
                log_file.write(line)
                
                # Capture the result payload
                if "__WORKER_RESULT__:" in line:
                    try:
                        json_str = line.split("__WORKER_RESULT__:", 1)[1].strip()
                        data = json.loads(json_str)
                        status = data.get('status', 'OK')
                        bfs_feasible = data.get('feasible')
                        corpus_path = data.get('corpus_path')
                    except:
                        pass

            if not timed_out:
                process.wait()
                if process.returncode != 0 and status == 'UNKNOWN':
                    status = f"CRASH (Exit {process.returncode})"

    except Exception as e:
        status = f"FAILED: {e}"

    duration = time.perf_counter() - start_time

    # Save the summary report
    json_data = {
        'seed': seed,
        'status': status,
        'feasible': bfs_feasible,
        'corpus_path': corpus_path,
        'time_s': duration
    }
    
    safe_f = str(bfs_feasible).lower() if bfs_feasible is not None else "none"
    try:
        with open(LOG_DIR / f"seed_{seed}_feasible_{safe_f}.json", 'w', encoding='utf-8') as f:
            json.dump(json_data, f, indent=2)
    except:
        pass

    symbol = '✅' if bfs_feasible else ('❌' if bfs_feasible is False else '❌')
    # Print a single completion line to keep the console clean.
    return f"[Seed {seed}] {symbol} Time: {duration:.1f}s | RAM: {psutil.virtual_memory().percent}%"


# === Step 3: Main execution flow (ThreadPoolExecutor manages subprocesses) ===

if __name__ == "__main__":
    print("-" * 50)
    print(f"Starting HPC Batch Runner")
    print(f"Range: {start_seed} -> {max_seed}")
    print(f"Max Parallel Workers: {KERN_NUM}")
    print(f"Logs: {LOG_DIR.resolve()}")
    print("-" * 50)

    total_start_time = time.perf_counter()

    # Use ThreadPoolExecutor because the manager mostly waits on subprocess I/O.
    # The actual compute workload runs inside subprocesses.
    with concurrent.futures.ThreadPoolExecutor(max_workers=KERN_NUM) as executor:
        future_to_seed = {
            executor.submit(run_single_seed, seed): seed 
            for seed in range(start_seed, max_seed + 1)
        }
        
        finished_count = 0
        total_tasks = len(future_to_seed)

        try:
            for future in concurrent.futures.as_completed(future_to_seed):
                seed = future_to_seed[future]
                try:
                    result_msg = future.result()
                    finished_count += 1
                    print(f"[{finished_count}/{total_tasks}] {result_msg}")
                except Exception as exc:
                    print(f"[Seed {seed}] Exception: {exc}")
                
                # Periodically trigger GC in the main process
                if finished_count % 10 == 0:
                    gc.collect()
        except KeyboardInterrupt:
            print("\nStopping manager... (Ctrl+C pressed)")
            executor.shutdown(wait=False, cancel_futures=True)

    print("-" * 50)
    print(f"All seeds processed in {time.perf_counter() - total_start_time:.2f}s")

    if TEMP_SCRIPT_PATH.exists():
        try:
            TEMP_SCRIPT_PATH.unlink()
        except:
            pass

Python version: 3.13.9 (v3.13.9:8183fa5e3f7, Oct 14 2025, 10:27:13) [Clang 16.0.0 (clang-1600.0.26.6)]
HPC Mode: Enabled
Target Core/Worker Count: 8
[Info] Jupyter environment detected. Ignoring CLI arguments (sys.argv).
Extracting code from /Users/bowen/PycharmProjects/MP-OPT/MP-OPT.ipynb to _temp_worker_script_hpc.py...
--------------------------------------------------
Starting HPC Batch Runner
Range: 1 -> 100
Max Parallel Workers: 8
Logs: /Users/bowen/PycharmProjects/MP-OPT/Log
--------------------------------------------------
[1/100] [Seed 3] ❌ Time: 2.4s | RAM: 65.4%
[2/100] [Seed 2] ❌ Time: 2.4s | RAM: 65.1%
[3/100] [Seed 7] ❌ Time: 2.5s | RAM: 65.1%
[4/100] [Seed 1] ❌ Time: 2.6s | RAM: 65.1%
[5/100] [Seed 9] ❌ Time: 2.3s | RAM: 66.0%
[6/100] [Seed 4] ❌ Time: 53.9s | RAM: 63.6%
[7/100] [Seed 6] ❌ Time: 100.5s | RAM: 63.3%
[8/100] [Seed 15] ❌ Time: 2.8s | RAM: 64.1%
[9/100] [Seed 14] ✅ Time: 89.2s | RAM: 64.4%
[10/100] [Seed 17] ❌ Time: 2.5s | RAM: 65.5%
[11/100] [Seed 18] ❌ Tim